# Query expansion on the hybrid retriever (LongMemEval_s)

Work kernel for memory-recall-bench.

In [ ]:
!pip install -q "memory-recall-bench[embeddings] @ git+https://github.com/Deepanjal-s/memory-recall-bench.git"

In [ ]:
import json, time, urllib.request
from collections import defaultdict
from pathlib import Path
import pandas as pd, torch
from memory_recall_bench import load_longmemeval, run
from memory_recall_bench.embeddings import EmbeddingCache, RecentPlusHybrid, sentence_transformers_encoder

EXP = json.loads('{"What degree did I graduate with?": ["graduation degree", "what degree did I earn", "graduated with what degree", "my university degree"], "How long is my daily commute to work?": ["daily commute to work", "time from home to office", "commute duration", "how long it takes to get to work"], "Where did I redeem a $5 coupon on coffee creamer?": ["redeem $5 coupon coffee creamer", "coffee creamer coupon redemption", "where did I use coupon for coffee creamer"], "What play did I attend at the local community theater?": ["local community theater play attendance", "theater performance I went to last summer", "name of play I saw at community theater", "event at community theater in October 2023"], "What is the name of the playlist I created on Spotify?": ["Spotify playlist I made", "name of playlist I created", "playlist I created on Spotify"], "What was my last name before I changed it?": ["last name before name change", "name change history", "previous last name", "old last name"], "Where do I take yoga classes?": ["yoga classes near me", "local yoga studios", "yoga class locations in [city name]", "best yoga studios in [neighborhood name]"], "What color did I repaint my bedroom walls?": ["repaint bedroom walls", "color of bedroom walls", "bedroom wall paint color", "when I repainted my bedroom walls"], "When did I volunteer at the local animal shelter\'s fundraising dinner?": ["volunteer at animal shelter fundraising dinner", "animal shelter event volunteer", "fundraising dinner volunteer date", "local animal shelter volunteer event"], "Where did I buy my new tennis racket from?": ["buy tennis racket", "where to buy tennis racket", "new tennis racket purchase", "tennis racket store name"], "What was my previous occupation?": ["previous job", "occupation history", "what did I work at", "my last job title"], "How much did I spend on a designer handbag?": ["spend on designer handbag", "price of designer handbag", "cost of handbag purchase", "designer handbag transaction"], "How many playlists do I have on Spotify?": ["Spotify playlists", "list of playlists", "my Spotify playlists", "playlists I created"], "Where did I attend for my study abroad program?": ["study abroad program location", "where I went for study abroad", "study abroad program country", "study abroad university name"], "What was the discount I got on my first purchase from the new clothing brand?": ["discount on first purchase", "new clothing brand", "first purchase", "clothing brand discount"], "How long did it take me to assemble the IKEA bookshelf?": ["IKEA bookshelf assembly time", "IKEA bookshelf assembly instructions", "how long to assemble IKEA bookshelf", "IKEA bookshelf build time"], "What did I buy for my sister\'s birthday gift?": ["birthday gift for sister", "gift purchase for sister", "what I bought for sister\'s birthday", "sister\'s birthday present"], "What speed is my new internet plan?": ["new internet plan speed"], "How many shirts did I pack for my 5-day trip to Costa Rica?": ["shirts packed for trip to Costa Rica", "day trip to Costa Rica packing list", "Costa Rica trip items packed", "what clothes did I pack for Costa Rica vacation"], "What was my previous stance on spirituality?": ["spirituality", "my thoughts on spirituality", "beliefs about spirituality", "views on spirituality"], "What brand are my favorite running shoes?": ["favorite running shoes", "running shoes brand", "shoes I wore in 2022", "running gear I bought last year"], "What certification did I complete last month?": ["certification completed last month", "list of certifications I earned", "what certification did I finish in May", "did I get a certification in June"], "How many bikes do I own?": ["bikes I have owned", "list of bikes I purchased", "bike inventory from 2022", "how many bikes I bought in 2023"], "What breed is my dog?": ["dog breed", "my dog\'s breed", "what breed is my dog", "dog breed information"], "How many largemouth bass did I catch on my fishing trip to Lake Michigan?": ["fishing trip to Lake Michigan", "largemouth bass caught", "Lake Michigan fishing trip", "bass caught during fishing trip"], "How many amateur comedians did I watch perform at the open mic night?": ["open mic night amateur comedians", "amateur comedian performance at open mic night"], "What is the name of my cat?": ["name of my cat"], "How old was I when my grandma gave me the silver necklace?": ["silver necklace", "grandma", "age when received silver necklace"], "What is my preferred gin-to-vermouth ratio for a classic gin martini?": ["gin martini recipe", "preferred gin-to-vermouth ratio", "classic gin martini mix", "how to make a gin martini"], "How much RAM did I upgrade my laptop to?": ["laptop RAM upgrade", "RAM upgrade amount", "how much RAM did I install on my laptop", "upgrade my laptop RAM to what size"], "How much is the painting of a sunset worth in terms of the amount I paid for it?": ["painting of a sunset price I paid for it", "amount I paid for sunset painting", "value of sunset painting I bought"], "Where did I attend my cousin\'s wedding?": ["cousin\'s wedding location", "wedding year", "where I went to cousin\'s wedding", "name of cousin and wedding event"], "Where did I complete my Bachelor\'s degree in Computer Science?": ["Bachelor\'s degree in Computer Science", "University where I earned my Bachelor\'s degree", "Computer Science degree completion location", "My undergraduate university for Computer Science"], "How long did it take to move to the new apartment?": ["move to new apartment", "new apartment move date", "apartment relocation time", "when did we move to the new apartment?"], "What type of cocktail recipe did I try last weekend?": ["cocktail recipe last weekend", "margarita recipe I tried", "whiskey cocktail I made", "mixer I used for cocktail last weekend"], "What type of rice is my favorite?": ["favorite rice", "rice preference", "what rice did you like best", "type of rice you preferred"], "Who did I have a conversation with about destiny?": ["conversation about destiny", "chat with someone on destiny", "talked to a person about destiny"], "Where did I buy my new bookshelf from?": ["bookshelf purchase", "where I bought bookshelf", "buying a bookshelf", "bookshelf store name"], "What time do I stop checking work emails and messages?": ["when did I stop checking work emails", "what time did I last check work messages", "did I stop checking work emails at 7 pm", "did I stop checking work messages after 6 pm"], "How many copies of my favorite artist\'s debut album were released worldwide?": ["debut album by [artist name] release quantity", "worldwide copies of [artist name] first album", "release details of [artist name] debut album"], "How long have I been collecting vintage cameras?": ["vintage cameras collection start date", "first vintage camera bought", "when I purchased my first vintage camera"], "Where did I go on a week-long trip with my family?": ["week-long trip with family", "family vacation location", "trip destination with family", "where I went on vacation with family"], "What did I bake for my niece\'s birthday party?": ["bake birthday cake for niece", "birthday party recipe niece", "baked dessert for niece\'s birthday"], "What is the name of the music streaming service have I been using lately?": ["music streaming service I used last month", "Spotify", "Apple Music", "YouTube Music"], "How long did I wait for the decision on my asylum application?": ["asylum application decision date", "date of asylum application submission", "when was my asylum application decided", "how long between submitting asylum application and decision"], "What type of action figure did I buy from a thrift store?": ["action figure bought at thrift store", "thrift store purchase of action figure", "action figure from used store", "what action figure did I get at a thrift store"], "How long was I in Japan for?": ["Japan trip duration", "time spent in Japan", "how long I stayed in Japan", "Japan visit length"], "Where did I attend the Imagine Dragons concert?": ["Imagine Dragons concert location", "concert I attended with Imagine Dragons", "where I went to see Imagine Dragons", "Imagine Dragons concert date and place"], "How much screen time have I been averaging on Instagram per day?": ["Instagram screen time daily average", "Instagram usage log", "daily Instagram time tracking", "Instagram app usage statistics"], "What type of bulb did I replace in my bedside lamp?": ["bedside lamp bulb replacement", "type of bulb I used", "bulb replacement in bedroom lamp"], "What size is my new Samsung TV?": ["Samsung TV size", "new Samsung TV dimensions", "Samsung TV model and size", "TV screen size from recent chat"], "What book am I currently reading?": ["current book I am reading"], "How many skeins of worsted weight yarn did I find in my stash?": ["yarn stash worsted weight skeins"], "How many hours did I spend watching documentaries on Netflix last month?": ["watched documentaries on Netflix", "Netflix documentary viewing time", "documentary hours watched"], "What time do I usually get home from work on weeknights?": ["home time from work", "weeknight work commute", "work ending time", "when I get home from work nights"], "What is my ethnicity?": ["my ethnicity", "ethnic background", "race information", "nationality history"], "How much time do I dedicate to practicing guitar every day?": ["guitar practice time", "daily guitar practice duration", "how long I practiced guitar yesterday", "guitar practice routine last week"], "Where does my sister Emily live?": ["Emily\'s address", "Where Emily lives", "Emily\'s home location", "Emily\'s city and state"], "How long did Alex marinate the BBQ ribs in special sauce?": ["marinate BBQ ribs with special sauce", "Alex marinate ribs", "special sauce recipe for ribs", "BBQ ribs marinade time"], "Who gave me a new stand mixer as a birthday gift?": ["birthday gift", "stand mixer", "gift from someone"], "What brand of shampoo do I currently use?": ["shampoo brand I use last month", "shampoo I bought in June", "what shampoo did I buy at the pharmacy", "shampoo I switched to in October"], "What health issue did I initially think was just a cold?": ["cold I thought was just a regular cold", "symptoms I thought were just a cold", "feeling sick and thought it was a cold", "initially thought was just a cold"], "Where did I meet Sophia?": ["meeting with Sophia", "when did I first see Sophia", "where was I when I met Sophia", "Sophia and I went to the caf\\u00e9 on March 15"], "What game did I finally beat last weekend?": ["game I finally beat", "last weekend gaming session", "beat a game", "game victory last weekend"], "What is the name of my hamster?": ["hamster name", "my hamster\'s name", "what is the name of my pet hamster", "name of the hamster I had"], "How long have I been collecting vintage films?": ["vintage films collection start date", "first vintage film I bought", "when I began collecting old movies", "first movie I purchased as part of my collection"], "What did I bake for my uncle\'s birthday party?": ["bake birthday cake for uncle", "birthday party recipe", "baked something for uncle\'s birthday"], "How long was I in Korea for?": ["trip to Korea", "time spent in Korea", "Korea visit duration", "how long I stayed in Korea"], "How much time do I dedicate to practicing violin every day?": ["violin practice time", "daily violin practice duration", "how long I practiced violin each day"], "What did my dad gave me as a birthday gift?": ["birthday gift", "gift from dad", "what did dad give me", "my dad\'s birthday present"], "How many items of clothing do I need to pick up or return from a store?": ["clothing items to pick up from store", "clothing items to return from store"], "How many projects have I led or am currently leading?": ["projects I led", "projects I am currently leading"], "How many model kits have I worked on or bought?": ["model kits", "kit purchases", "model kit projects", "model kit collections"], "How many days did I spend on camping trips in the United States this year?": ["camping trip in the United States", "days spent camping", "camping trip duration", "U.S. camping trip this year"], "How many weeks did it take me to watch all the Marvel Cinematic Universe movies and the main Star Wars films?": ["Marvel Cinematic Universe movies", "main Star Wars films", "watching Marvel movies", "watching Star Wars films"], "How many plants did I acquire in the last month?": ["plants acquired", "purchase of plants", "added plants to collection", "new plants in last month"], "How much total money have I spent on bike-related expenses since the start of the year?": ["bike expenses", "spending on bikes", "bike purchases", "bike-related costs"], "How many hours in total did I spend driving to my three road trip destinations combined?": ["driving hours to destination 1", "driving hours to destination 2", "driving hours to destination 3"], "How many different doctors did I visit?": ["doctor", "medical visit", "health appointment", "hospital visit"], "What time did I go to bed on the day before I had a doctor\'s appointment?": ["time I went to bed before doctor\'s appointment", "doctor\'s appointment date", "when I went to bed the night before doctor\'s appointment"], "How many different types of citrus fruits have I used in my cocktail recipes?": ["citrus fruits in cocktail recipes", "types of citrus used in drinks", "cocktail recipes with orange", "cocktail recipes with lemon"], "How many movie festivals that I attended?": ["movie festivals I went to", "list of film festivals I attended", "festival events I participated in", "film festival trips I made"], "How many days did I take social media breaks in total?": ["social media break", "days without social media", "social media break duration", "when I took a social media break"], "How many tanks do I currently have, including the one I set up for my friend\'s kid?": ["tanks I set up for my friend\'s kid", "tanks I currently have"], "What is the total amount I spent on luxury items in the past few months?": ["spend on luxury items", "luxury items purchase history", "total amount spent on watches and jewelry", "expenses on designer clothing and accessories"], "How many hours have I spent playing games in total?": ["played games", "game session", "game time", "game duration"], "How many weddings have I attended in this year?": ["weddings attended this year", "list of weddings in 2024", "events marked as weddings in current year", "wedding invitations received this year"], "How many babies were born to friends and family members in the last few months?": ["babies born to friends in last 3 months", "babies born to family members in last 3 months"], "How many pieces of furniture did I buy, assemble, sell, or fix in the past few months?": ["furniture I bought in the past few months", "furniture I assembled in the past few months", "furniture I sold in the past few months", "furniture I fixed in the past few months"], "How many times did I bake something in the past two weeks?": ["bake", "past two weeks", "recipe", "cake"], "How many different museums or galleries did I visit in the month of February?": ["museums visited in February", "galleries visited in February"], "How many different cuisines have I learned to cook or tried out in the past few months?": ["cuisine I learned to cook last month", "cuisine I tried out in August", "cuisine I attempted in September", "recipes I made in October"], "How many properties did I view before making an offer on the townhouse in the Brookside neighborhood?": ["properties viewed", "offer on townhouse", "Brookside neighborhood"], "How many different types of food delivery services have I used recently?": ["food delivery services used", "Uber Eats", "Grubhub", "DoorDash"], "How many hours of jogging and yoga did I do last week?": ["jogging hours last week", "yoga hours last week"], "Which social media platform did I gain the most followers on over the past month?": ["followers on Instagram", "followers on Twitter", "followers on Facebook", "followers on TikTok"], "Which grocery store did I spend the most money at in the past month?": ["grocery store spending last month", "amount spent at grocery store", "highest spending at grocery store", "grocery store purchase history past month"], "How much more did I spend on accommodations per night in Hawaii compared to Tokyo?": ["accommodations cost in Hawaii", "accommodations cost in Tokyo", "average nightly cost Hawaii", "average nightly cost Tokyo"], "How many different art-related events did I attend in the past month?": ["art event", "painting class", "gallery opening", "creative workshop"], "What is the average age of me, my parents, and my grandparents?": ["my age", "my parents\' age", "my grandparents\' age"], "How many doctor\'s appointments did I go to in March?": ["doctor\'s appointment March", "appointments in March", "medical visit March"], "How many days did it take for me to receive the new remote shutter release after I ordered it?": ["order new remote shutter release", "received new remote shutter release"], "How many health-related devices do I use in a day?": ["smartwatch", "fitness tracker", "blood pressure monitor", "glucose meter"], "How many fish are there in total in both of my aquariums?": ["fish in aquarium 1", "fish in aquarium 2"], "How many fitness classes do I attend in a typical week?": ["fitness class attendance", "weekly fitness classes", "how many times I go to gym", "go to yoga class"], "How many days did it take for my laptop backpack to arrive after I bought it?": ["laptop backpack purchase date", "laptop backpack delivery date"], "How many pieces of jewelry did I acquire in the last two months?": ["jewelry purchases last two months", "list of jewelry items bought in past 60 days", "items acquired related to jewelry in recent month", "purchase history of necklaces, rings, earrings, bracelets in last two months"], "How much money did I raise in total through all the charity events I participated in?": ["charity event fundraising amount", "total money raised from charity events", "fundraising total for all events I attended"], "How many projects have I been working on simultaneously, excluding my thesis?": ["projects", "thesis", "working on multiple projects"], "How many musical instruments do I currently own?": ["list of musical instruments I have owned", "inventory of instruments I bought in 2022", "items in my home collection related to music", "musical instruments I purchased last year"], "How many bikes did I service or plan to service in March?": ["bikes serviced in March", "bikes planned for service in March"], "How much money did I raise for charity in total?": ["charity fundraiser money raised", "total amount of money raised", "donation total for charity event", "how much I raised for charity"], "How many days did I spend participating in faith-related activities in December?": ["faith-related activities", "December", "days spent on activities", "participation in religious events"], "How many kitchen items did I replace or fix?": ["kitchen items replaced", "kitchen items fixed"], "How many times did I ride rollercoasters across all the events I attended from July to October?": ["ride rollercoaster July", "ride rollercoaster August", "ride rollercoaster September", "ride rollercoaster October"], "How much total money did I spend on attending workshops in the last four months?": ["money spent on workshops", "workshops attendance cost", "spending on workshops last 4 months"], "How many days did I spend in total traveling in Hawaii and in New York City?": ["traveling in hawaii", "traveling in new york city"], "How many days did I spend attending workshops, lectures, and conferences in April?": ["workshops in April", "lectures in April", "conferences in April"], "How many rare items do I have in total?": ["list of rare items I collected", "count of rare items in my inventory", "rare items I acquired in 2023", "rare items from the game collection"], "What is the total amount of money I earned from selling my products at the markets?": ["total money earned from selling products at markets", "sales amount from market transactions", "earnings from product sales at local markets", "money made from selling items at market events"], "How many magazine subscriptions do I currently have?": ["magazine subscriptions list", "current magazine subscriptions", "subscriptions I have active", "list of my magazine subscriptions"], "How many online courses have I completed in total?": ["online courses completed", "list of courses I finished", "courses I finished in 2023", "courses I completed in 2024"], "How many music albums or EPs have I purchased or downloaded?": ["music albums purchased", "music EPs downloaded", "downloads history", "purchased music files"], "How many years in total did I spend in formal education from high school to the completion of my Bachelor\'s degree?": ["high school years", "bachelor\'s degree duration", "formal education timeline from high school to bachelor\'s", "total years in school from high school to bachelor\'s degree"], "How many total pieces of writing have I completed since I started writing again three weeks ago, including short stories, poems, and pieces for the writing challenge?": ["writing challenge submissions", "short stories completed", "poems written", "writing started three weeks ago"], "How many graduation ceremonies have I attended in the past three months?": ["graduation ceremony", "past three months", "my graduation events", "graduation attendance"], "How many times did I bake egg tarts in the past two weeks?": ["bake egg tarts", "past two weeks", "egg tarts recipe", "baking session with egg tarts"], "How many different museums or galleries did I visit in December?": ["museums visited in December", "galleries visited in December"], "How many fish are there in my 30-gallon tank?": ["gallon fish tank contents", "fish in tank", "number of fish in aquarium", "aquarium fish count"], "How many days did it take for my iPad case to arrive after I bought it?": ["iPad case purchase date", "iPad case delivery date"], "How many days did I spend in total traveling in Hawaii and in Seattle?": ["traveling in hawaii", "traveling in seattle"], "How many years in total did I spend in formal education from high school to the completion of my Master\'s degree?": ["high school graduation year", "master\'s degree completion year"], "Can you recommend some resources where I can learn more about video editing?": ["video editing tutorials", "best video editing software", "how to edit videos on a computer", "free video editing tools"], "Can you suggest some accessories that would complement my current photography setup?": ["photography setup accessories", "tripod for photography", "lens filter for photography", "camera bag for photography"], "Can you recommend some recent publications or conferences that I might find interesting?": ["recent publications I have recommended", "conferences I have mentioned recently", "publications from the last 6 months", "conferences in the past year that I suggested"], "Can you suggest a hotel for my upcoming trip to Miami?": ["upcoming trip to Miami", "hotel recommendations for Miami", "Miami travel accommodations", "best hotels in Miami"], "Can you recommend some interesting cultural events happening around me this weekend?": ["cultural events this weekend", "events near me this weekend", "interesting events in my area this weekend"], "Can you recommend a show or movie for me to watch tonight?": ["recently watched shows or movies", "recommendations for entertainment tonight", "what shows or movies did you suggest last week", "favorite movie or show suggestions from past chats"], "Can you suggest some activities that I can do in the evening?": ["evening activities I enjoyed last week", "what did I do in the evening during summer 2023", "recommendations for relaxing evening routines", "activities I tried last month after work"], "My kitchen\'s becoming a bit of a mess again. Any tips for keeping it clean?": ["kitchen mess", "clean kitchen tips", "kitchen clutter solutions", "how to organize kitchen"], "I\'ve been struggling with my slow cooker recipes. Any advice on getting better results?": ["slow cooker recipes", "slow cooker tips", "slow cooker cooking mistakes", "slow cooker recipe failures"], "I\'ve been thinking about ways to stay connected with my colleagues. Any suggestions?": ["chat about team meetings", "discussions with colleagues", "messages about staying in touch", "talk about work collaboration"], "What should I serve for dinner this weekend with my homegrown ingredients?": ["homegrown tomatoes", "homegrown zucchini", "homegrown herbs", "homegrown garlic"], "I\'ve been feeling a bit stuck with my paintings lately. Do you have any ideas on how I can find new inspiration?": ["painting ideas", "art inspiration", "creative blocks", "new painting projects"], "I\'ve been thinking about making a cocktail for an upcoming get-together, but I\'m not sure which one to choose. Any suggestions?": ["cocktail recipes", "get-together drinks", "party cocktail ideas", "mixology suggestions"], "I\'ve been having trouble with the battery life on my phone lately. Any tips?": ["phone battery life", "battery issues", "phone battery problems", "battery drains fast"], "I\'ve been feeling like my chocolate chip cookies need something extra. Any advice?": ["chocolate chip cookies recipe", "cookies with extra flavor", "how to make cookies taste better", "additions to chocolate chip cookies"], "I\'m thinking of inviting my colleagues over for a small gathering. Any tips on what to bake?": ["baking tips for small gatherings", "what to bake for colleagues", "easy baked treats for office events", "desserts for small indoor gatherings"], "I was thinking about rearranging the furniture in my bedroom this weekend. Any tips?": ["furniture rearrangement tips for bedroom", "bedroom layout ideas", "how to organize bedroom furniture", "best ways to rearrange bedroom furniture"], "I\'m getting excited about my visit to the music store this weekend. Any tips on what to look for in a new guitar?": ["guitar shopping tips", "best guitars for beginners", "what to look for in a new guitar", "music store guitar recommendations"], "I was thinking of trying a new coffee creamer recipe. Any recommendations?": ["coffee creamer recipe", "new coffee creamer recipe ideas", "creamer recipe for coffee", "coffee creamer substitutions"], "I\'ve been sneezing quite a bit lately. Do you think it might be my living room?": ["sneezing in living room", "living room allergies", "sneezing symptoms recent", "allergens in living room"], "I\'ve been feeling nostalgic lately. Do you think it would be a good idea to attend my high school reunion?": ["high school reunion", "my high school name", "reunion event date", "past conversations about school events"], "I\'m trying to decide whether to buy a NAS device now or wait. What do you think?": ["NAS device recommendations", "storage solutions for home use", "best NAS devices in 2023", "home network setup advice"], "I am planning another theme park weekend; do you have any suggestions?": ["theme park weekend ideas", "past theme park recommendations", "fun activities for a weekend getaway", "best theme parks for families"], "I\'m planning my meal prep next week, any suggestions for new recipes?": ["meal prep recipes", "new recipes to try", "recipe ideas for weeknight meals", "healthy meal prep options"], "I\'m planning a trip to Denver soon. Any suggestions on what to do there?": ["Denver attractions", "Denver activities", "Denver tourist spots", "Things to do in Denver"], "I\'ve got some free time tonight, any documentary recommendations?": ["documentary recommendations", "free time tonight", "nighttime activities", "what documentaries did you suggest recently"], "I noticed my bike seems to be performing even better during my Sunday group rides. Could there be a reason for this?": ["bike performance", "Sunday group rides", "ride on Sunday", "group ride with friends"], "Can you suggest some useful accessories for my phone?": ["phone accessories", "smartphone cases", "phone chargers", "phone stands"], "Can you suggest some activities I can do during my commute to work?": ["commute to work activities", "work commute ideas", "things to do on the way to work", "daily commute hobbies"], "I\\u2019m a bit anxious about getting around Tokyo. Do you have any helpful tips?": ["tips for getting around Tokyo", "public transportation in Tokyo", "Tokyo subway guide", "how to use Tokyo metro"], "What is the total distance of the hikes I did on two consecutive weekends?": ["hikes I did on weekend of June 10-11", "hikes I did on weekend of June 17-18"], "How many pages do I have left to read in \'The Nightingale\'?": ["The Nightingale book pages left to read", "pages remaining in The Nightingale", "how many pages left in The Nightingale book"], "For my daily commute, how much more expensive was the taxi ride compared to the train fare?": ["daily commute taxi cost", "daily commute train fare", "taxi vs train fare comparison", "commute expenses taxi and train"], "What was the approximate increase in Instagram followers I experienced in two weeks?": ["Instagram followers count", "follower growth two weeks ago", "increase in followers over 14 days"], "How many antique items did I inherit or acquire from my family members?": ["antique items inherited from family", "antique items acquired from family member", "list of antique items received from grandparents", "antique items given by parents or relatives"], "What is the total cost of the new food bowl, measuring cup, dental chews, and flea and tick collar I got for Max?": ["total cost of food bowl for Max", "cost of measuring cup for Max", "price of dental chews for Max", "expense of flea and tick collar for Max"], "How much cashback did I earn at SaveMart last Thursday?": ["cashback SaveMart last Thursday", "SaveMart cashback transaction last Thursday"], "What is the difference in price between my luxury boots and the similar pair found at the budget store?": ["price of my luxury boots", "price of the similar pair at the budget store"], "What percentage of packed shoes did I wear on my last trip?": ["packed shoes", "last trip", "wore shoes"], "When did I submit my research paper on sentiment analysis?": ["research paper submission sentiment analysis", "sentiment analysis paper due date", "submitted research paper on sentiment analysis"], "What percentage discount did I get on the book from my favorite author?": ["book discount", "favorite author sale", "percentage off book purchase"], "Did I receive a higher percentage discount on my first order from HelloFresh, compared to my first UberEats order?": ["HelloFresh first order discount percentage", "UberEats first order discount percentage"], "What is the total number of episodes I\'ve listened to from \'How I Built This\' and \'My Favorite Murder\'?": ["How I Built This\\" episodes listened to", "My Favorite Murder\\" episodes listened to"], "How many plants did I initially plant for tomatoes and cucumbers?": ["initial plant count for tomatoes", "initial plant count for cucumbers"], "How many hours do I work in a typical week during peak campaign seasons?": ["hours worked per week", "campaign season", "peak campaign season", "typical work week"], "How much older am I than the average age of employees in my department?": ["my age", "average age of employees in my department"], "What was the total number of people reached by my Facebook ad campaign and Instagram influencer collaboration?": ["Facebook ad campaign reach", "Instagram influencer collaboration reach"], "How many Marvel movies did I re-watch?": ["re-watch marvel movies", "marvel movie re-watch list", "how many marvel movies did i re-watch", "marvel film re-watch session"], "How much did I save on the designer handbag at TK Maxx?": ["search for \\"designer handbag\\" purchase at TK Maxx", "search for \\"TK Maxx\\" discount or savings on handbag", "search for \\"handbag\\" price and savings in past chat with TK Maxx"], "What is the total number of goals and assists I have in the recreational indoor soccer league?": ["total goals and assists in recreational indoor soccer league", "recreational indoor soccer league stats", "goals and assists in indoor soccer league", "soccer league performance record"], "How much did I spend on car wash and parking ticket?": ["car wash expense", "parking ticket cost"], "How many sports have I played competitively in the past?": ["competitive sports", "sports I played", "list of sports I participated in", "activities involving competition I did"], "What are the two hobbies that led me to join online communities?": ["hobbies that led me to join online communities", "online communities I joined because of photography", "online communities I joined because of coding"], "How old was I when Alex was born?": ["my birth year", "Alex\'s birth year"], "How many points do I need to earn to redeem a free skincare product at Sephora?": ["Sephora points required for free skincare product", "Sephora redemption points skincare product", "free skincare product Sephora points needed"], "What is the total number of days I spent in Japan and Chicago?": ["days spent in Japan", "days spent in Chicago"], "What is the minimum amount I could get if I sold the vintage diamond necklace and the antique vanity?": ["vintage diamond necklace sale price", "antique vanity sale price"], "What percentage of the countryside property\'s price is the cost of the renovations I plan to do on my current house?": ["cost of renovations on current house", "price of countryside property", "renovations planned for current house", "percentage of property price for renovations"], "What is the total cost of Lola\'s vet visit and flea medication?": ["vet visit cost for Lola", "flea medication cost for Lola"], "How much more did I have to pay for the trip after the initial quote?": ["initial quote for trip", "final cost of trip", "difference between initial quote and final trip cost"], "What is the total number of lunch meals I got from the chicken fajitas and lentil soup?": ["lunch meal with chicken fajitas", "lunch meal with lentil soup"], "How much did I spend on each coffee mug for my coworkers?": ["coffee mug purchase", "spending on coffee mugs", "cost of coffee mugs for coworkers", "mug order for coworkers"], "At which university did I present a poster on my thesis research?": ["poster presentation", "thesis research", "university presentation", "academic conference"], "How long have I been working in my current role?": ["when did I start my current job", "what is my current role", "when did I begin working at [company name]"], "How much more was the pre-approval amount than the final sale price of the house?": ["pre-approval amount for house purchase", "final sale price of house", "house purchase transaction details"], "What is the total cost of the car cover and detailing spray I purchased?": ["car cover purchase", "detailing spray purchase"], "What is the total distance I covered in my four road trips?": ["total distance of four road trips", "road trip 1 distance", "road trip 2 distance", "road trip 3 distance"], "What is the total time it takes I to get ready and commute to work?": ["get ready routine time", "commute time to work", "morning routine duration", "travel time to office"], "How much more miles per gallon was my car getting a few months ago compared to now?": ["car mileage a few months ago", "current car mileage", "miles per gallon comparison", "fuel efficiency change"], "What is the total number of online courses I\'ve completed?": ["online courses completed", "list of courses I finished", "courses I completed in 2023", "total number of courses I finished"], "How much did I save on the Jimmy Choo heels?": ["Jimmy Choo heels purchase amount", "savings on Jimmy Choo heels", "how much I saved on Jimmy Choo heels"], "How many years will I be when my friend Rachel gets married?": ["Rachel\'s wedding year", "my birth year", "Rachel\'s marriage date"], "How much faster did I finish the 5K run compared to my previous year\'s time?": ["K run time last year", "K run time this year"], "How many dinner parties have I attended in the past month?": ["dinner party past month", "attended dinner party", "dinner party invitation received", "dinner party with friends last month"], "How much did I spend on gifts for my sister?": ["spend on gifts for sister", "gift purchase for sister", "gifts given to sister", "money spent on sister\'s birthday gift"], "What time did I reach the clinic on Monday?": ["time i reached the clinic", "Monday clinic visit", "when did i go to the clinic on Monday"], "What is the total weight of the new feed I purchased in the past two months?": ["new feed purchase", "past two months feed purchase", "total weight of feed bought", "feed weight purchased last month"], "What percentage of leadership positions do women hold in the my company?": ["leadership positions in my company", "percentage of women in leadership at my company"], "How much will I save by taking the train from the airport to my hotel instead of a taxi?": ["train fare from airport to hotel", "taxi fare from airport to hotel"], "What is the total number of views on my most popular videos on YouTube and TikTok?": ["total views on most popular YouTube videos", "total views on most popular TikTok videos"], "How many years older is my grandma than me?": ["grandma\'s birth year", "my birth year"], "How many years older am I than when I graduated from college?": ["my graduation year from college", "my current age", "year I started college"], "What is the total amount I spent on gifts for my coworker and brother?": ["total amount spent on gifts for coworker", "total amount spent on gifts for brother"], "What is the total number of comments on my recent Facebook Live session and my most popular YouTube video?": ["Facebook Live session comments count", "YouTube video comments count", "most popular YouTube video title", "recent Facebook Live session date"], "How many days a week do I attend fitness classes?": ["fitness class attendance", "days I went to gym", "how often I did workouts", "fitness class schedule"], "What is the total amount I spent on the designer handbag and high-end skincare products?": ["total amount spent on designer handbag", "total amount spent on high-end skincare products"], "How much more money did I raise than my initial goal in the charity cycling event?": ["charity cycling event goal amount", "charity cycling event final amount raised", "amount raised in charity cycling event", "initial goal for charity cycling event"], "What is the average GPA of my undergraduate and graduate studies?": ["average GPA undergraduate studies", "average GPA graduate studies"], "How many minutes did I exceed my target time by in the marathon?": ["marathon target time", "marathon actual time", "time difference marathon"], "What was the page count of the two novels I finished in January and March?": ["page count of novel finished in January", "page count of novel finished in March"], "What is the total number of siblings I have?": ["number of siblings", "siblings in my family", "how many brothers and sisters I have"], "How much have I made from selling eggs this month?": ["sales from eggs", "earnings from egg sales", "egg selling revenue", "money made from selling eggs this month"], "How much discount will I get on my next purchase at FreshMart?": ["discount offered at FreshMart", "recent purchase at FreshMart", "FreshMart promo code", "past shopping at FreshMart"], "How much earlier do I wake up on Fridays compared to other weekdays?": ["wake up time Friday", "wake up time Monday", "wake up time Tuesday", "wake up time Wednesday"], "How many fun runs did I miss in March due to work commitments?": ["fun runs in March", "work commitments in March", "missed events in March", "run events and work schedule March"], "How many pages do I have left to read in \'Sapiens\'?": ["pages left in Sapiens", "book Sapiens reading progress", "how many pages left in Sapiens", "Sapiens book chapter completion"], "How many plants did I initially plant for tomatoes and chili peppers?": ["initially plant tomatoes", "initially plant chili peppers"], "What is the total cost of my recently purchased headphones and the iPad?": ["headphones purchase date and price", "iPad purchase date and price"], "At which university did I present a poster for my undergrad course research project?": ["poster presentation university", "undergrad research project presentation", "course research project presentation", "university where I presented poster"], "How old will Rachel be when I get married?": ["Rachel\'s age", "user\'s marriage date"], "How much will I save by taking the bus from the airport to my hotel instead of a taxi?": ["bus fare from airport to hotel", "taxi fare from airport to hotel"], "How many days passed between my visit to the Museum of Modern Art (MoMA) and the \'Ancient Civilizations\' exhibit at the Metropolitan Museum of Art?": ["MoMA visit date", "Metropolitan Museum of Art Ancient Civilizations exhibit date"], "Which three events happened in the order from first to last: the day I helped my friend prepare the nursery, the day I helped my cousin pick out stuff for her baby shower, and the day I ordered a customized phone case for my friend\'s birthday?": ["day I helped my friend prepare the nursery", "day I helped my cousin pick out stuff for her baby shower", "day I ordered a customized phone case for my friend\'s birthday"], "How many weeks ago did I meet up with my aunt and receive the crystal chandelier?": ["meet up with aunt", "received crystal chandelier"], "How many months have passed since I participated in two charity events in a row, on consecutive days?": ["charity event consecutive days", "charity event first day", "charity event second day"], "How many days passed between the day I started playing along to my favorite songs on my old keyboard and the day I discovered a bluegrass band?": ["day I started playing along to my favorite songs on my old keyboard", "day I discovered a bluegrass band"], "How many months have passed since I last visited a museum with a friend?": ["museum visit with friend", "last museum visit date", "friend museum trip", "museum outing with friend"], "How many weeks ago did I attend the friends and family sale at Nordstrom?": ["friends and family sale Nordstrom", "attend Nordstrom sale", "Nordstrom sale week ago"], "Which event happened first, my cousin\'s wedding or Michael\'s engagement party?": ["cousin\'s wedding date", "michael\'s engagement party date"], "How many days ago did I attend the Maundy Thursday service at the Episcopal Church?": ["Maundy Thursday service Episcopal Church", "Episcopal Church service date", "Maundy Thursday service attendance", "days ago I went to Episcopal Church service"], "How many days ago did I attend a baking class at a local culinary school when I made my friend\'s birthday cake?": ["baking class local culinary school", "made friend\'s birthday cake"], "How many days passed between the day I started watering my herb garden and the day I harvested my first batch of fresh herbs?": ["watering herb garden", "harvested first batch of fresh herbs"], "How many weeks ago did I start using the cashback app \'Ibotta\'?": ["Ibotta app start date", "when did I first use Ibotta", "Ibotta app first usage", "when did I begin using Ibotta"], "How many days had passed since I finished reading \'The Seven Husbands of Evelyn Hugo\' when I attended the book reading event at the local library, where the author of \'The Silent Patient\' is discussing her latest thriller novel?": ["finished reading The Seven Husbands of Evelyn Hugo", "book reading event at local library", "author of The Silent Patient discussing latest thriller novel"], "How many days did I spend on my solo camping trip to Yosemite National Park?": ["solo camping trip Yosemite National Park", "Yosemite National Park camping trip duration", "days spent camping in Yosemite", "Yosemite camping trip start and end date"], "What is the order of the three trips I took in the past three months, from earliest to latest?": ["trip to paris", "trip to san francisco", "trip to seattle"], "How many months ago did I attend the Seattle International Film Festival?": ["Seattle International Film Festival", "attend film festival", "months ago film festival attendance"], "How many days ago did I buy a smoker?": ["buy smoker", "smoker purchase", "smoker bought", "when did I buy smoker"], "What is the order of the three events: \'I signed up for the rewards program at ShopRite\', \'I used a Buy One Get One Free coupon on Luvs diapers at Walmart\', and \'I redeemed $12 cashback for a $10 Amazon gift card from Ibotta\'?": ["I signed up for the rewards program at ShopRite", "I used a Buy One Get One Free coupon on Luvs diapers at Walmart", "I redeemed $12 cashback for a $10 Amazon gift card from Ibotta"], "How many weeks in total do I spent on reading \'The Nightingale\' and listening to \'Sapiens: A Brief History of Humankind\' and \'The Power\'?": ["weeks spent reading The Nightingale", "weeks spent listening to Sapiens: A Brief History of Humankind", "weeks spent listening to The Power"], "How many days passed between the day I cancelled my FarmFresh subscription and the day I did my online grocery shopping from Instacart?": ["cancel FarmFresh subscription", "online grocery shopping Instacart"], "How many weeks passed between the day I bought my new tennis racket and the day I received it?": ["bought new tennis racket", "received new tennis racket"], "How many days ago did I meet Emma?": ["meet Emma"], "What is the order of the six museums I visited from earliest to latest?": ["museums I visited", "list of museums I went to", "order of museums I visited", "museum visits timeline"], "How many days passed between the day I attended the gardening workshop and the day I planted the tomato saplings?": ["gardening workshop", "planted tomato saplings"], "How many days passed between the day I received feedback about my car\'s suspension and the day I tested my new suspension setup?": ["feedback about my car\'s suspension", "tested my new suspension setup"], "How many days had passed since I started taking ukulele lessons when I decided to take my acoustic guitar to the guitar tech for servicing?": ["ukulele lessons start date", "acoustic guitar servicing date"], "How many weeks passed between the time I sold homemade baked goods at the Farmers\' Market for the last time and the time I participated in the Spring Fling Market?": ["sold homemade baked goods at the Farmers\' Market", "participated in the Spring Fling Market"], "What is the order of the sports events I watched in January?": ["sports events watched in January", "list of events I watched in January", "January sports events I attended", "events I watched in January related to sports"], "How many days did it take me to finish \'The Nightingale\' by Kristin Hannah?": ["The Nightingale\\" by Kristin Hannah reading completion date", "The Nightingale\\" by Kristin Hannah finished reading", "reading The Nightingale\\" end date", "Kristin Hannah The Nightingale\\" start and end date"], "What is the order of the three sports events I participated in during the past month, from earliest to latest?": ["sports event January 10", "sports event January 25", "sports event February 3"], "How many weeks had passed since I recovered from the flu when I went on my 10th jog outdoors?": ["flu recovery date", "th jog outdoors"], "What is the order of the concerts and musical events I attended in the past two months, starting from the earliest?": ["concerts in the past two months", "musical events I attended", "list of events from last month", "events starting from earliest date"], "How many days passed between the day I finished reading \'The Nightingale\' and the day I started reading \'The Hitchhiker\'s Guide to the Galaxy\'?": ["day I finished reading The Nightingale", "day I started reading The Hitchhiker\'s Guide to the Galaxy"], "Which mode of transport did I use most recently, a bus or a train?": ["transport mode used in last chat", "bus ride mentioned in recent conversation", "train travel referenced in past messages"], "How many weeks have I been taking sculpting classes when I invested in my own set of sculpting tools?": ["sculpting classes started", "invested in sculpting tools", "when did I buy sculpting tools", "how long after buying tools did I start sculpting classes"], "How many days ago did I attend a networking event?": ["networking event", "attend event", "days ago event", "event attendance"], "What is the order of airlines I flew with from earliest to latest before today?": ["airlines I flew with", "flight history by airline", "list of airlines used in past trips", "flying with specific airlines timeline"], "How many days passed between the day I repotted the previous spider plant and the day I gave my neighbor, Mrs. Johnson, a few cuttings from my spider plant?": ["repotted spider plant", "gave cuttings to Mrs. Johnson"], "How many days passed between the day I replaced my spark plugs and the day I participated in the Turbocharged Tuesdays auto racking event?": ["spark plugs replacement", "Turbocharged Tuesdays auto racking event"], "How many weeks ago did I attend a bird watching workshop at the local Audubon society?": ["bird watching workshop", "Audubon society", "local Audubon society", "workshop at Audubon society"], "How many months passed between the completion of my undergraduate degree and the submission of my master\'s thesis?": ["undergraduate degree completion date", "master\'s thesis submission date"], "How many months ago did I attend the photography workshop?": ["photography workshop", "attend workshop", "months ago workshop", "when did I go to photography workshop"], "Which event did I participate in first, the charity gala or the charity bake sale?": ["charity gala participation", "charity bake sale participation"], "How many days ago did I watch the Super Bowl?": ["Super Bowl watch date", "Super Bowl viewing log", "when did i watch super bowl", "Super Bowl game day note"], "How many days ago did I launch my website when I signed a contract with my first client?": ["launch website", "signed contract with first client"], "How many days ago did I go on a whitewater rafting trip in the Oregon mountains?": ["whitewater rafting trip Oregon mountains", "rafting trip in Oregon mountains", "Oregon mountains rafting adventure", "whitewater rafting in Oregon mountains"], "How many days ago did I harvest my first batch of fresh herbs from the herb garden kit?": ["harvest first batch fresh herbs herb garden kit", "herb garden kit first harvest date"], "How many weeks ago did I attend the \'Summer Nights\' festival at Universal Studios Hollywood?": ["Summer Nights festival Universal Studios Hollywood", "Universal Studios Hollywood festival attendance", "Summer Nights festival date Universal Studios Hollywood"], "How many days ago did I participate in the 5K charity run?": ["K charity run", "charity run participation", "days ago I ran 5K", "when did I do the 5K run"], "Which event happened first, my participation in the #PlankChallenge or my post about vegan chili recipe?": ["#PlankChallenge", "vegan chili recipe post"], "How many days passed between the day I fixed my mountain bike and the day I decided to upgrade my road bike\'s pedals?": ["fixed my mountain bike", "decided to upgrade my road bike\'s pedals"], "Who graduated first, second and third among Emma, Rachel and Alex?": ["graduation order Emma Rachel Alex", "list of students who graduated in 2022", "who graduated first second third Emma Rachel Alex"], "How many days ago did I read the March 15th issue of The New Yorker?": ["read The New Yorker March 15"], "Which book did I finish a week ago?": ["book I finished a week ago", "reading log from last week", "completed book last week", "finished reading a book a week ago"], "I mentioned participating in a sports event two weeks ago. What was the event?": ["sports event two weeks ago", "event I attended two weeks ago", "participated in a sports event two weeks ago"], "I received a piece of jewelry last Saturday from whom?": ["jewelry received last Saturday", "sender of jewelry last Saturday"], "What charity event did I participate in a month ago?": ["charity event I went to a month ago", "event I attended last month", "volunteer event I participated in one month ago"], "Who did I go with to the music event last Saturday?": ["music event last Saturday", "group went to music event last Saturday", "who attended music event last Saturday", "went to music event with whom last Saturday"], "What gardening-related activity did I do two weeks ago?": ["gardening activity two weeks ago", "plants I grew two weeks ago", "gardening project two weeks ago", "time spent gardening two weeks ago"], "What was the airline that I flied with on Valentine\'s day?": ["flight on Valentine\'s Day", "airline I used for my trip on February 14", "flight details for February 14th", "Valentine\'s Day travel booking"], "I mentioned that I participated in an art-related event two weeks ago. Where was that event held at?": ["art event two weeks ago location", "event I attended two weeks ago venue", "art-related event location two weeks ago"], "Which bike did I fixed or serviced the past weekend?": ["bike repair", "service bike", "fixed bike", "bike maintenance past weekend"], "What was the the life event of one of my relatives that I participated in a week ago?": ["relative event", "participated in event", "week ago event", "family gathering last week"], "Who did I meet with during the lunch last Tuesday?": ["lunch last tuesday", "meeting last tuesday", "who I met at lunch last tuesday"], "What is the artist that I started to listen to last Friday?": ["artist I started listening to last Friday"], "I mentioned cooking something for my friend a couple of days ago. What was it?": ["cooking for friend", "what did I cook for friend", "mentioned cooking recipe for friend", "prepared meal for friend a few days ago"], "What was the significant buisiness milestone I mentioned four weeks ago?": ["business milestone", "four weeks ago", "significant event", "company expansion"], "What did I do with Rachel on the Wednesday two months ago?": ["Wednesday two months ago with Rachel", "activity with Rachel on Wednesday", "what I did with Rachel that Wednesday"], "I mentioned visiting a museum two months ago. Did I visit with a friend or not?": ["museum visit two months ago with friend", "museum visit two months ago alone"], "I mentioned an investment for a competition four weeks ago? What did I buy?": ["investment for competition four weeks ago", "what did I buy in the competition investment", "competition investment purchase four weeks ago"], "What kitchen appliance did I buy 10 days ago?": ["kitchen appliance bought 10 days ago", "appliance purchase 10 days ago", "what did I buy in the kitchen 10 days ago"], "Where did I attend the religious activity last week?": ["religious activity last week", "location of religious event", "where I went last week for a religious gathering"], "What was the social media activity I participated 5 days ago?": ["social media posts I made 5 days ago", "posts I shared on Instagram 5 days ago", "messages I sent on Twitter 5 days ago", "comments I left on Facebook 5 days ago"], "What was the first issue I had with my new car after its first service?": ["first service of my new car", "issues with new car after service", "what went wrong with my car service", "first problem after car service"], "Which event did I attend first, the \'Effective Time Management\' workshop or the \'Data Analysis using Python\' webinar?": ["Effective Time Management\\" workshop", "Data Analysis using Python\\" webinar"], "Which vehicle did I take care of first in February, the bike or the car?": ["vehicle I took care of in February", "bike I cared for in February", "car I maintained in February"], "Which device did I got first, the Samsung Galaxy S22 or the Dell XPS 13?": ["Samsung Galaxy S22", "Dell XPS 13"], "How many days before the team meeting I was preparing for did I attend the workshop on \'Effective Communication in the Workplace\'?": ["workshop on Effective Communication in the Workplace", "team meeting preparation", "days between workshop and team meeting"], "How many days had passed between the Sunday mass at St. Mary\'s Church and the Ash Wednesday service at the cathedral?": ["Sunday mass at St. Mary\'s Church", "Ash Wednesday service at the cathedral"], "How many days did it take for me to find a house I loved after starting to work with Rachel?": ["find house with Rachel", "start working with Rachel", "days between starting work with Rachel and finding house I loved"], "Which seeds were started first, the tomatoes or the marigolds?": ["seeds started with tomatoes", "seeds started with marigolds"], "How many days had passed between the Hindu festival of Holi and the Sunday mass at St. Mary\'s Church?": ["Holi festival date", "Sunday mass St. Mary\'s Church date"], "How many days before the \'Rack Fest\' did I participate in the \'Turbocharged Tuesdays\' event?": ["Turbocharged Tuesdays\\" event date", "Rack Fest\\" event date"], "Which pair of shoes did I clean last month?": ["clean shoes last month", "shoes cleaning activity last month"], "Which event happened first, the purchase of the coffee maker or the malfunction of the stand mixer?": ["purchase of coffee maker", "malfunction of stand mixer"], "How many days had passed between the \'Walk for Hunger\' event and the \'Coastal Cleanup\' event?": ["Walk for Hunger\\" event date", "Coastal Cleanup\\" event date"], "Which item did I purchase first, the dog bed for Max or the training pads for Luna?": ["purchase dog bed for Max", "purchase training pads for Luna"], "Which task did I complete first, fixing the fence or trimming the goats\' hooves?": ["fixing the fence", "trimming the goats\' hooves"], "How many charity events did I participate in before the \'Run for the Cure\' event?": ["charity event before Run for the Cure", "Run for the Cure event date", "list of charity events I attended", "participated in charity event prior to Run for the Cure"], "How long have I been working before I started my current job at NovaTech?": ["previous job before NovaTech", "work history prior to joining NovaTech", "employment timeline before NovaTech", "job at Company X before NovaTech"], "Which book did I finish reading first, \'The Hate U Give\' or \'The Nightingale\'?": ["book I finished reading first", "The Hate U Give", "The Nightingale"], "Which device did I set up first, the smart thermostat or the mesh network system?": ["smart thermostat setup", "mesh network system setup"], "How many months ago did I book the Airbnb in San Francisco?": ["book Airbnb San Francisco", "Airbnb booking San Francisco", "San Francisco Airbnb reservation"], "How long did I take to finish \'The Seven Husbands of Evelyn Hugo\' and \'The Nightingale\' combined?": ["The Seven Husbands of Evelyn Hugo completion time", "The Nightingale completion time"], "What was the date on which I attended the first BBQ event in June?": ["barbecue event June first", "date of first BBQ in June"], "How many days before I bought the iPhone 13 Pro did I attend the Holiday Market?": ["iPhone 13 Pro purchase date", "Holiday Market attendance date"], "Which show did I start watching first, \'The Crown\' or \'Game of Thrones\'?": ["start watching The Crown", "start watching Game of Thrones"], "How long had I been a member of \'Book Lovers Unite\' when I attended the meetup?": ["membership start date Book Lovers Unite", "date of Book Lovers Unite meetup"], "How long had I been watching stand-up comedy specials regularly when I attended the open mic night at the local comedy club?": ["watching stand-up comedy specials regularly", "attended open mic night", "local comedy club"], "What time do I wake up on Tuesdays and Thursdays?": ["wake up time Tuesday", "wake up time Thursday"], "How many days had passed since I bought my Adidas running shoes when I realized one of the shoelaces on my old Converse sneakers had broken?": ["when I bought Adidas running shoes", "when I realized Converse shoelace broke"], "Which event happened first, the road trip to the coast or the arrival of the new prime lens?": ["road trip to the coast", "arrival of the new prime lens"], "Which airline did I fly with the most in March and April?": ["flight airline March", "flight airline April"], "How long had I been bird watching when I attended the bird watching workshop?": ["bird watching workshop", "bird watching", "when I attended the bird watching workshop"], "Which trip did I take first, the one to Europe with family or the solo trip to Thailand?": ["trip to Europe with family", "solo trip to Thailand"], "Which device did I set up first, the smart thermostat or the new router?": ["smart thermostat setup", "new router setup"], "Which health issue did I deal with first, the persistent cough or the skin tag removal?": ["persistent cough", "skin tag removal"], "How old was I when I moved to the United States?": ["when did I move to the united states", "what year did I move to the united states", "how old was I in the year I moved to the united states"], "How long had I been using the new area rug when I rearranged my living room furniture?": ["area rug installation date", "living room furniture rearrangement date"], "How many days before my best friend\'s birthday party did I order her gift?": ["order gift for best friend", "best friend\'s birthday party", "date of best friend\'s birthday party"], "Which trip did the narrator take first, the solo trip to Europe or the family road trip across the American Southwest?": ["solo trip to Europe", "family road trip across the American Southwest"], "How long did I use my new binoculars before I saw the American goldfinches returning to the area?": ["binoculars usage duration", "American goldfinches return date", "when I first saw American goldfinches with binoculars"], "Who did I meet first, Mark and Sarah or Tom?": ["meet Mark", "meet Sarah", "meet Tom"], "How many weeks have I been accepted into the exchange program when I started attending the pre-departure orientation sessions?": ["accepted into exchange program", "started attending pre-departure orientation sessions"], "Which group did I join first, \'Page Turners\' or \'Marketing Professionals\'?": ["join Page Turners", "join Marketing Professionals"], "How long had I been taking guitar lessons when I bought the new guitar amp?": ["guitar lessons start date", "bought new guitar amp date"], "Which streaming service did I start using most recently?": ["recent streaming service I started using", "most recent streaming platform I switched to", "when did I begin using Netflix", "when did I start using Hulu"], "How many months before my anniversary did Rachel get engaged?": ["Rachel engagement", "anniversary date", "engagement month", "months between engagement and anniversary"], "Which event happened first, the narrator losing their phone charger or the narrator receiving their new phone case?": ["narrator lost phone charger", "narrator received new phone case"], "Who did I meet first, the woman selling jam at the farmer\'s market or the tourist from Australia?": ["farmer\'s market jam seller", "tourist from Australia"], "Which event happened first, the meeting with Rachel or the pride parade?": ["meeting with rachel", "pride parade"], "Which gift did I buy first, the necklace for my sister or the photo album for my mom?": ["gift for sister necklace", "gift for mom photo album"], "Which event did I participate in first, the volleyball league or the charity 5K run to raise money for a local children\'s hospital?": ["volleyball league participation", "charity 5K run children\'s hospital"], "Which event happened first, my attendance at a cultural festival or the start of my Spanish classes?": ["attendance at a cultural festival", "start of Spanish classes"], "Which project did I start first, the Ferrari model or the Japanese Zero fighter plane model?": ["project Ferrari model start date", "project Japanese Zero fighter plane model start date"], "Who became a parent first, Rachel or Alex?": ["Rachel\'s first child was born in May 2022", "Alex had a child in January 2023"], "How many days had passed between the day I bought a gift for my brother\'s graduation ceremony and the day I bought a birthday gift for my best friend?": ["gift for brother\'s graduation", "birthday gift for best friend"], "Which task did I complete first, fixing the fence or purchasing three cows from Peter?": ["fixing the fence", "purchasing three cows from Peter"], "How long have I been working before I started my current job at Google?": ["previous job before Google", "work history before joining Google", "employment timeline before Google", "last job before Google"], "When did I book the Airbnb in Sacramento?": ["book Airbnb Sacramento", "Airbnb reservation Sacramento", "Sacramento Airbnb booking date"], "How many days before I bought my iPad did I attend the Holiday Market?": ["iPad purchase date", "Holiday Market attendance date"], "Which project did I start first, the Ferrari model or the Porsche 991 Turbo S model?": ["project start date Ferrari model", "project start date Porsche 991 Turbo S model"], "Who became a parent first, Tom or Alex?": ["Tom\'s first child was born in 2020", "Alex\'s first child was born in 2021"], "What was my personal best time in the charity 5K run?": ["personal best time charity 5K run", "charity 5K run time record", "K run performance history", "my fastest 5K time event"], "How many Korean restaurants have I tried in my city?": ["Korean restaurant", "restaurant visit", "Korean food experience", "tried Korean food in city"], "Where did Rachel move to after her recent relocation?": ["Rachel relocation", "Rachel moved to", "Rachel new city", "Rachel moved to where"], "What was the amount I was pre-approved for when I got my mortgage from Wells Fargo?": ["mortgage pre-approval amount Wells Fargo", "Wells Fargo mortgage approval amount", "pre-approved loan amount Wells Fargo", "Wells Fargo home loan approval figure"], "How often do I attend yoga classes to help with my anxiety?": ["yoga class attendance anxiety", "when did I go to yoga class for anxiety", "how many times did I attend yoga class last month", "did I go to yoga class in October for anxiety relief"], "Is my mom using the same grocery list method as me?": ["grocery list method used by mom", "grocery list method I used", "mom\'s grocery list entries", "my grocery list entries"], "How many hours have I spent on my abstract ocean sculpture?": ["hours spent on abstract ocean sculpture", "abstract ocean sculpture project duration", "time spent working on abstract ocean sculpture"], "How many bikes do I currently own?": ["bikes I have owned", "list of bikes I purchased", "bike inventory from last year", "how many bikes I bought in 2023"], "What day of the week do I take a cocktail-making class?": ["cocktail-making class date", "day of week cocktail class", "when did I take a cocktail-making class"], "Where did I go on my most recent family trip?": ["most recent family trip location", "family vacation destination", "where did I go with family last summer", "family trip to national park"], "Where do I initially keep my old sneakers?": ["old sneakers", "first place to store sneakers", "where I put my sneakers when I first got them"], "How many short stories have I written since I started writing regularly?": ["short stories", "writing regularly", "started writing"], "How many pages of \'A Short History of Nearly Everything\' have I read so far?": ["pages read in A Short History of Nearly Everything", "reading progress for A Short History of Nearly Everything", "how many pages of A Short History of Nearly Everything completed", "A Short History of Nearly Everything reading log"], "How many engineers do I lead when I just started my new role as Senior Software Engineer? How many engineers do I lead now?": ["Senior Software Engineer role start date", "number of engineers I lead as Senior Software Engineer", "current number of engineers I lead"], "How many titles are currently on my to-watch list?": ["to-watch list", "my to-watch list items", "watch list titles", "list of shows and movies to watch"], "How many stars do I need to reach the gold level on my Starbucks Rewards app?": ["Starbucks Rewards gold level stars required", "Starbucks Rewards app level requirements", "how many stars to get gold on Starbucks Rewards"], "How long have I been using my Fitbit Charge 3?": ["Fitbit Charge 3 purchase date", "Fitbit Charge 3 setup date", "Fitbit Charge 3 first use date"], "How many sessions of the bereavement support group did I attend?": ["bereavement support group attendance", "sessions attended in bereavement support group", "how many times did I go to bereavement support group", "bereavement group meeting count"], "What type of camera lens did I purchase most recently?": ["recent camera lens purchase", "camera lens bought last month", "what lens did I buy for my camera", "camera lens purchase history"], "How often do I see my therapist, Dr. Smith?": ["therapy session with Dr. Smith", "frequency of visits to Dr. Smith", "how often I see Dr. Smith in past messages", "Dr. Smith appointment schedule"], "How many MCU films did I watch in the last 3 months?": ["MCU film watch list", "films watched in last 3 months", "watched Avengers movie", "watched Black Panther movie"], "How many of Emma\'s recipes have I tried out?": ["Emma\'s recipes", "recipes I tried", "tried out Emma\'s recipes"], "What company is Rachel, an old colleague from my previous company, currently working at?": ["Rachel\'s current company", "Rachel\'s last job", "Rachel\'s workplace after leaving previous company"], "For the coffee-to-water ratio in my French press, did I switch to more water per tablespoon of coffee, or less?": ["coffee-to-water ratio French press", "French press recipe change", "adjustment to water amount in French press", "more water per tablespoon coffee French press"], "Where is the painting \'Ethereal Dreams\' by Emma Taylor currently hanging?": ["painting Ethereal Dreams by Emma Taylor", "location of painting Ethereal Dreams", "where is painting Ethereal Dreams currently displayed", "Emma Taylor Ethereal Dreams exhibition location"], "How many times have I worn my new black Converse Chuck Taylor All Star sneakers?": ["wore black Converse Chuck Taylor All Star sneakers", "Converse sneakers", "black Converse"], "How many episodes of the Science series have I completed on Crash Course?": ["Crash Course Science episodes completed", "Crash Course Science series", "list of Crash Course Science episodes I watched", "how many Crash Course Science episodes I finished"], "Do I go to the gym more frequently than I did previously?": ["gym visits last month", "gym visits last quarter", "frequency of gym workouts in January", "number of gym sessions in March"], "How many new postcards have I added to my collection since I started collecting again?": ["new postcards added to collection", "started collecting postcards"], "How many videos of Corey Schafer\'s Python programming series have I completed so far?": ["Corey Schafer Python programming series videos", "videos completed in Corey Schafer Python course", "Python programming series by Corey Schafer progress", "Corey Schafer Python course completion status"], "How long have I been living in my current apartment in Harajuku?": ["when did I move into the apartment in Harajuku", "what is the address of my current apartment in Harajuku", "when did I first rent the apartment in Harajuku"], "What brand of BBQ sauce am I currently obsessed with?": ["BBQ sauce I used last week", "favorite BBQ sauce from June 2023", "BBQ sauce recommendation in May 2024", "BBQ sauce I mentioned in July 2023"], "How long have my parents been staying with me in the US?": ["parents moving to US", "when did my parents arrive in the US", "my parents staying with me in the US", "parents relocation to US"], "How long have I been sticking to my daily tidying routine?": ["daily tidying routine", "tidying routine start date", "when did I start daily tidying", "last time I did daily tidying"], "How long have I had my cat, Luna?": ["cat Luna", "when did I get cat Luna", "Luna cat purchase date", "when I got Luna the cat"], "How many Instagram followers do I currently have?": ["Instagram followers count", "my Instagram account followers", "how many followers does my Instagram have", "Instagram follower number"], "How many dozen eggs do we currently have stocked up in our refrigerator?": ["eggs in refrigerator", "stocked up eggs", "current egg inventory", "how many eggs in fridge"], "How many issues of National Geographic have I finished reading?": ["National Geographic magazine", "list of National Geographic issues I read", "National Geographic issues completed", "reading National Geographic in 2022"], "Did I finish reading \'The Nightingale\' by Kristin Hannah?": ["The Nightingale by Kristin Hannah reading log", "Did I finish reading The Nightingale", "When did I finish reading The Nightingale", "Did I read The Nightingale by Kristin Hannah"], "Where did I get my guitar serviced?": ["guitar service location", "when was my guitar last serviced", "who serviced my guitar", "where did I take my guitar for repair"], "What is my current highest score in Ticket to Ride?": ["Ticket to Ride score", "highest score Ticket to Ride", "current Ticket to Ride score"], "How many tops have I bought from H&M so far?": ["H&M tops purchased", "list of H&M tops I bought", "tops from H&M in my chat history"], "How much did I earn at the Downtown Farmers Market on my most recent visit?": ["Downtown Farmers Market earnings", "most recent visit Downtown Farmers Market", "earnings at Downtown Farmers Market"], "How many times have I tried making a Negroni at home since my friend Emma showed me how to make it?": ["made Negroni at home", "Emma showed me how to make Negroni"], "What time do I usually go to the gym?": ["gym visit time", "when did I go to the gym", "gym schedule", "go to gym time"], "How many Crash Course videos have I watched in the past few weeks?": ["Crash Course video watched", "Crash Course playlist past few weeks", "Crash Course video view count", "Crash Course watch history recent"], "Where am I planning to stay for my birthday trip to Hawaii?": ["birthday trip to Hawaii accommodation", "Hawaii vacation hotel plans", "where I\'m staying for my birthday trip to Hawaii"], "How many different species of birds have I seen in my local park?": ["birds", "local park", "species of birds", "park bird sightings"], "What was my previous frequent flyer status on United Airlines before I got the current status?": ["United Airlines frequent flyer status", "previous frequent flyer status", "frequent flyer status before current United status"], "What was my previous personal best time for the charity 5K run?": ["personal best time charity 5K run", "charity 5K run time record", "K run personal best time", "my fastest 5K run time"], "How many autographed baseballs have I added to my collection in the first three months of collection?": ["autographed baseball added to collection", "first three months of collection", "baseball collection started", "added autographed baseball to collection"], "How often do I play tennis with my friends at the local park previously? How often do I play now?": ["play tennis with friends local park", "play tennis now"], "How much time do I dedicate to coding exercises each day?": ["coding exercises daily time", "time spent on coding exercises", "daily coding exercise routine", "how long I code each day"], "What day of the week did I meet with my previous language exchange tutor Juan?": ["meeting with Juan language exchange tutor"], "How many pre-1920 American coins do I have in my collection?": ["American coins", "pre-1920 coins", "coin collection", "coins before 1920"], "How many free night\'s stays can I redeem at any Hilton property with my accumulated points?": ["free nights stays Hilton points redemption", "Hilton points free night stays policy", "Hilton property free night redemption", "Hilton points how many free nights"], "How much weight have I lost since I started going to the gym consistently?": ["weight lost", "started going to the gym", "consistent gym attendance"], "How many followers do I have on Instagram now?": ["Instagram followers count", "my Instagram account followers", "followers on Instagram as of last month", "how many people follow me on Instagram"], "What is my current record in the recreational volleyball league?": ["recreational volleyball league score", "my volleyball game results", "volleyball league standings", "volleyball match score recent"], "How many projects have I completed since starting painting classes?": ["painting classes projects completed", "projects started after beginning painting classes", "list of projects finished in painting classes", "completed painting class projects since first class"], "What type of vehicle model am I currently working on?": ["vehicle model", "current project", "car model", "work on vehicle"], "What was my previous goal for my Apex Legends level before I updated my goal?": ["Apex Legends level goal", "Apex Legends level update", "previous Apex Legends goal"], "How many largemouth bass did I catch with Alex on the earlier fishing trip to Lake Michigan before the 7/22 trip?": ["fishing trip with Alex Lake Michigan", "largemouth bass caught", "fishing trip before 7/22", "Lake Michigan fishing trip with Alex"], "What time do I wake up on Saturday mornings?": ["wake up time Saturday morning", "Saturday morning routine", "when do I wake up on Saturday", "wake up schedule Saturday"], "How many women are on the team led by my former manager Rachel?": ["team led by Rachel", "women on team", "Rachel\'s team members"], "Do I have a spare screwdriver for opening up my laptop?": ["screwdriver laptop", "laptop repair tools", "spare screwdriver in my toolkit", "used screwdriver to open laptop"], "How many times have I met up with Alex from Germany?": ["meeting with Alex from Germany", "chat with Alex Germany", "talked to Alex Germany", "met Alex Germany"], "Did I mostly recently increase or decrease the limit on the number of cups of coffee in the morning?": ["increase coffee cup limit", "decrease coffee cup limit"], "How many trips have I taken my Canon EOS 80D camera on?": ["Canon EOS 80D trips", "trips with Canon EOS 80D", "how many times did I take Canon EOS 80D on a trip"], "What new kitchen gadget did I invest in before getting the Air Fryer?": ["kitchen gadget bought before air fryer", "new kitchen appliance installed prior to air fryer purchase"], "How many Italian restaurants have I tried in my city?": ["Italian restaurant", "restaurant visit", "food outing", "dining in city"], "How many engineers do I lead when I just started my new role as Software Engineer Manager?": ["Software Engineer Manager role start date", "number of engineers reported under management at first role", "team size when first assigned as Software Engineer Manager", "initial team size as Software Engineer Manager"], "How often do I see Dr. Johnson?": ["search for \\"Dr. Johnson\\" in chat history", "look up when I saw Dr. Johnson last", "find messages about appointments with Dr. Johnson", "check chat entries with Dr. Johnson in 2023"], "How long have I been living in my current apartment in Shinjuku?": ["when did I move into my apartment in Shinjuku", "what is the address of my current apartment in Shinjuku", "when did I first rent the apartment in Shinjuku"], "How many autographed football have I added to my collection in the first three months of collection?": ["autographed football added to collection", "first three months of collection", "football collection entry", "add autographed football to collection"], "How often do I play table tennis with my friends at the local park?": ["table tennis game with friends at the park", "play table tennis at local park", "table tennis match at city park", "friends playing table tennis together at park"], "Before I purchased the gravel bike, do I have other bikes in addition to my mountain bike and my commuter bike?": ["purchased gravel bike", "own mountain bike", "own commuter bike", "bikes before gravel bike purchase"], "Where do I currently keep my old sneakers?": ["old sneakers", "where I last stored my old sneakers", "sneakers storage location", "current location of old sneakers"], "I\'m checking our previous chat about the shift rotation sheet for GM social media agents. Can you remind me what was the rotation for Admon on a Sunday?": ["shift rotation sheet GM social media agents", "Admon shift rotation Sunday"], "I\'m planning to visit Bandung again and I was wondering if you could remind me of the name of that restaurant in Cihampelas Walk that serves a great Nasi Goreng?": ["restaurant in Cihampelas Walk", "Nasi Goreng restaurant", "Bandung Cihampelas Walk restaurant", "name of restaurant that serves Nasi Goreng in Bandung"], "I\'m going back to our previous conversation about the children\'s book on dinosaurs. Can you remind me what color was the scaly body of the Plesiosaur in the image?": ["children\'s book dinosaurs plesiosaur color scaly body", "image of plesiosaur in children\'s book"], "I\'m planning to revisit Orlando. I was wondering if you could remind me of that unique dessert shop with the giant milkshakes we talked about last time?": ["giant milkshakes dessert shop Orlando", "Orlando dessert shop with giant milkshakes"], "Can you remind me of the name of the romantic Italian restaurant in Rome you recommended for dinner?": ["romantic Italian restaurant in Rome recommendation", "dinner restaurant in Rome Italy", "Italian restaurant in Rome for a date", "Rome restaurant suggestion for couple dinner"], "I remember you told me about the refining processes at CITGO\'s three refineries earlier. Can you remind me what kind of processes are used at the Lake Charles Refinery?": ["refining processes at CITGO Lake Charles Refinery", "CITGO Lake Charles Refinery operations", "types of refining processes used at Lake Charles Refinery"], "I\'m planning my trip to Amsterdam again and I was wondering, what was the name of that hostel near the Red Light District that you recommended last time?": ["hostel near Red Light District Amsterdam recommendation", "Amsterdam hostel recommendation last time"], "I think we discussed work from home jobs for seniors earlier. Can you remind me what was the 7th job in the list you provided?": ["work from home jobs for seniors", "list of 7th work from home job for seniors"], "In our previous chat, you suggested \'sexual compulsions\' and a few other options for alternative terms for certain behaviors. Can you remind me what the other four options were?": ["previous chat suggestions for alternative terms to sexual compulsions", "list of other terms besides sexual compulsions for behavioral patterns", "what were the four alternative terms you provided for sexual compulsions", "alternative phrases for compulsive behaviors in past conversation"], "I wanted to check back on our previous conversation about Netflix. I mentioned that I wanted to be able to access all seasons of old shows? Do you remember what show I used as an example, the one that only had the last season available?": ["Netflix", "old shows", "access all seasons", "show with only last season available"], "I wanted to follow up on our previous conversation about binaural beats for anxiety and depression. Can you remind me how many subjects were in the study published in the journal Music and Medicine that found significant reductions in symptoms of depression, anxiety, and stress?": ["binaural beats study Music and Medicine depression anxiety stress", "study in Music and Medicine with binaural beats and depression symptoms", "subjects in Music and Medicine journal binaural beats anxiety depression study"], "I was thinking about our previous conversation about the Fifth Album, and I was wondering if you could remind me what song you said best exemplified the band\'s growth and development as artists?": ["Fifth Album conversation", "song that best exemplified the band\'s growth", "mention of a specific song from the Fifth Album", "discussion about artistic development in the Fifth Album"], "I wanted to follow up on our previous conversation about front-end and back-end development. Can you remind me of the specific back-end programming languages you recommended I learn?": ["previous conversation front-end and back-end development", "back-end programming languages recommended", "specific back-end languages suggested"], "I was looking back at our previous conversation about Native American powwows and I was wondering, which traditional game did you say was often performed by skilled dancers at powwows?": ["powwow traditional game skilled dancers play", "games at Native American powwows", "traditional powwow games and dances"], "I\'m going back to our previous chat about the Lost Temple of the Djinn one-shot. Can you remind me how many mummies the party will face in the temple?": ["Lost Temple of the Djinn one-shot mummies count"], "I\'m planning to go back to the Natural Park of Moncayo mountain in Arag\\u00f3n and I was wondering, what was the name of that hiking trail you recommended that takes you through the park\'s most stunning landscapes and offers panoramic views of the surrounding mountainside?": ["hiking trail in Natural Park of Moncayo", "trail recommendation for Natural Park of Moncayo", "panoramic view hiking trail Moncayo mountain", "hiking route through Natural Park of Moncayo with mountain views"], "I was going through our previous conversation about The Library of Babel, and I wanted to confirm - what did Borges say about the center and circumference of the Library?": ["The Library of Babel Borges center and circumference", "Borges The Library of Babel center", "Borges The Library of Babel circumference"], "I\'m going back to our previous conversation about the grant aim page on molecular subtypes and endometrial cancer. Can you remind me what were the three objectives we outlined for the project?": ["grant aim page", "molecular subtypes", "endometrial cancer", "three objectives"], "I was thinking about our previous conversation about data privacy and security. You mentioned that companies use two-factor authentication to enhance security. Can you remind me what kind of two-factor authentication methods you were referring to?": ["two-factor authentication methods mentioned in our previous chat", "data privacy conversation with you", "companies using two-factor authentication for security"], "I\'m planning to visit the Vatican again and I was wondering if you could remind me of the name of that famous deli near the Vatican that serves the best cured meats and cheeses?": ["Vatican deli with cured meats and cheeses", "cured meats and cheeses near Vatican", "famous deli in Vatican City", "deli near St. Peter\'s Basilica with cheeses and meats"], "I\'m going back to our previous conversation about DIY home decor projects using recycled materials. Can you remind me what sealant you recommended for the newspaper flower vase?": ["DIY home decor projects with recycled materials", "newspaper flower vase sealant recommendation", "sealant for newspaper flower vase", "recycled materials project with newspaper vase"], "I\'m planning my trip to Speyer again and I wanted to confirm, what\'s the phone number of the Speyer tourism board that you provided me earlier?": ["phone number of Speyer tourism board", "Speyer tourism board contact information", "tourism board phone number Speyer"], "I was going through our previous chat and I wanted to clarify something about the prayer of beginners in Tanqueray\'s Spiritual Life treatise. Can you remind me which chapter of the second part discusses vocal prayer and meditation?": ["prayer of beginners Tanqueray Spiritual Life treatise", "vocal prayer Tanqueray Spiritual Life treatise", "meditation Tanqueray Spiritual Life treatise", "second part Tanqueray Spiritual Life treatise chapter"], "I was going through our previous conversation about the impact of the political climate in Catalonia on its literature and music. Can you remind me of the example you gave of a Spanish-Catalan singer-songwriter who supports unity between Catalonia and Spain?": ["Spanish-Catalan singer-songwriter who supports unity between Catalonia and Spain", "example of a singer from Catalonia who promotes unity with Spain", "music and literature in Catalonia during political changes", "political climate in Catalonia and its effect on Catalan artists"], "I remember you provided a list of 100 prompt parameters that I can specify to influence your output. Can you remind me what was the 27th parameter on that list?": ["list of 100 prompt parameters", "th parameter in prompt parameters list", "prompt parameters list with 100 items", "what is the 27th item in the 100 prompt parameters list"], "I\'m looking back at our previous conversation about building a cocktail bar. You recommended five bottles to make the widest variety of gin-based cocktails. Can you remind me what the fifth bottle was?": ["building a cocktail bar", "recommended five bottles of gin", "fifth bottle of gin for cocktails"], "I wanted to follow up on our previous conversation about mindfulness techniques. You mentioned some great resources for guided imagery exercises, can you remind me of the website that had free exercises like \'The Mountain Meditation\' and \'The Body Scan Meditation\'?": ["guided imagery exercises mindfulness", "The Mountain Meditation website", "The Body Scan Meditation free resources"], "I\'m looking back at our previous chess game and I was wondering, what was the move you made after 27. Kg2 Bd5+?": ["chess game move after 27. Kg2 Bd5+", "Kg2 Bd5+ response", "chess game with move 27. Kg2 and Bd5+"], "I was going through our previous conversation about atmospheric correction methods, and I wanted to confirm - you mentioned that 6S, MAJA, and Sen2Cor are all algorithms for atmospheric correction of remote sensing images. Can you remind me which one is implemented in the SIAC_GEE tool?": ["S algorithm in SIAC_GEE", "MAJA algorithm in SIAC_GEE", "Sen2Cor algorithm in SIAC_GEE"], "I was going through our previous conversation about making a classic French omelette, and I wanted to confirm - how many eggs did you say we need for the recipe?": ["recipe for classic French omelette", "number of eggs in French omelette recipe", "how many eggs for classic French omelette"], "I\'m planning another trip to New York City and I was wondering if you could remind me of that vegan eatery you recommended last time, the one with multiple locations throughout the city?": ["vegan eatery in New York City", "vegan restaurant with multiple locations in NYC", "recommendation for vegan restaurant in New York City", "vegan eatery with branches across New York City"], "I wanted to follow up on our previous conversation about the fusion breakthrough at Lawrence Livermore National Laboratory. Can you remind me who is the President\'s Chief Advisor for Science and Technology mentioned in the article?": ["fusion breakthrough Lawrence Livermore National Laboratory", "President\'s Chief Advisor for Science and Technology"], "I was going through our previous conversation about political propaganda and humor, and I was wondering if you could remind me of that Soviet cartoon you mentioned that mocked Western culture?": ["Soviet cartoon that mocked Western culture", "political propaganda and humor conversation", "cartoon about Western culture in Soviet Union", "Western culture satire in Soviet cartoons"], "I wanted to follow up on our previous conversation about natural remedies for dark circles under the eyes. You mentioned applying tomato juice mixed with lemon juice, how long did you say I should leave it on for?": ["tomato juice and lemon juice for dark circles under eyes", "duration for tomato and lemon juice on eyes", "how long to leave tomato and lemon juice on under eyes"], "I was going through our previous chat. Can you remind me of the name of the last venue you recommended in the list of popular venues in Portland for indie music shows?": ["last venue recommended for indie music shows in Portland", "popular venues in Portland for indie music shows"], "Can you remind me what was the average improvement in framerate when using the Hardware-Aware Modular Training (HAMT) agent in the \'To Adapt or Not to Adapt? Real-Time Adaptation for Semantic Segmentation\' submission?": ["HAMT agent average framerate improvement", "To Adapt or Not to Adapt? Real-Time Adaptation for Semantic Segmentation\\" submission", "framerate improvement HAMT agent"], "I wanted to follow up on our previous conversation about language learning apps. You mentioned a few options, and I was wondering if you could remind me of the one that uses mnemonics to help learners memorize words and phrases?": ["language learning apps with mnemonics", "mnemonics for language learning", "app that uses mnemonics to memorize words and phrases"], "I\'m looking back at our previous chat about the DHL Wellness Retreats campaign. Can you remind me how much was allocated for influencer marketing in the campaign plan?": ["DHL Wellness Retreats campaign influencer marketing budget", "DHL Wellness Retreats campaign plan influencer marketing allocation"], "I was going through our previous conversation and I was wondering if you could remind me of the two companies you mentioned that prioritize employee safety and well-being like Triumvirate?": ["companies that prioritize employee safety and well-being", "Triumvirate employee safety policies", "employee well-being initiatives in tech companies", "safety-focused companies mentioned in chat history"], "I wanted to follow up on our previous conversation about private sector businesses in Chaudhary. Can you remind me of the company that employs over 40,000 people in the rug-manufacturing industry?": ["private sector businesses in Chaudhary", "company with over 40,000 employees in rug manufacturing", "rug manufacturing industry company in Chaudhary"], "I\'m looking back at our previous conversation about the Bajimaya v Reward Homes Pty Ltd case. Can you remind me what year the construction of the house began?": ["Bajimaya v Reward Homes Pty Ltd construction start year"], "I wanted to follow up on our previous conversation about YouTube videos for workplace posture. Can you remind me of the Mayo Clinic video you recommended?": ["Mayo Clinic YouTube video workplace posture", "workplace posture YouTube video recommendation", "YouTube video for workplace posture Mayo Clinic"], "I was going through our previous chat and I was wondering, what was Andy wearing in the script you wrote for the comedy movie scene?": ["Andy\'s outfit in the comedy movie scene script", "comedy movie scene script with Andy\'s clothing details"], "I was looking back at our previous chat and I wanted to confirm, how many times did the Chiefs play the Jaguars at Arrowhead Stadium?": ["Chiefs vs Jaguars Arrowhead Stadium", "Chiefs game against Jaguars at Arrowhead", "Arrowhead Stadium Chiefs Jaguars game history", "Chiefs and Jaguars game at Arrowhead Stadium"], "I was thinking back to our previous conversation about the Radiation Amplified zombie, and I was wondering if you remembered what we finally decided to name it?": ["Radiation Amplified zombie name decision", "previous conversation Radiation Amplified zombie", "name we chose for Radiation Amplified zombie"], "I was looking back at our previous conversation about buying unique engagement rings directly from designers. Can you remind me of the Instagram handle of the UK-based designer who works with unusual gemstones?": ["Instagram handle of UK-based designer who works with unusual gemstones", "engagement ring designer UK unusual gemstones", "buying engagement rings directly from designers Instagram", "unique engagement rings designer with gemstones UK"], "I\'m trying to recall what the designation on my jumpsuit was that helped me find the file number in the records room?": ["jumpsuit designation", "file number records room", "designation on jumpsuit"], "I\'m going back to our previous conversation about music theory. You mentioned some online resources for learning music theory. Can you remind me of the website you recommended for free lessons and exercises?": ["website recommended for free music theory lessons and exercises", "free music theory lessons and exercises online", "online resource for learning music theory you suggested", "music theory free lessons and exercises website"], "I\'m looking back at our previous conversation about the Seco de Cordero recipe from Ancash. You mentioned using a light or medium-bodied beer, but I was wondering if you could remind me what type of beer you specifically recommended?": ["Seco de Cordero recipe Ancash beer recommendation", "light or medium-bodied beer Seco de Cordero recipe"], "I wanted to follow up on our previous conversation about fracking in the Marcellus Shale region. You mentioned that some states require fracking companies to monitor groundwater quality at nearby wells before drilling and for a certain period after drilling is complete. Can you remind me which state you mentioned as an example that has this requirement?": ["fracking Marcellus Shale groundwater monitoring state requirement", "state with fracking groundwater quality monitoring before and after drilling"], "I\'m looking back at our previous conversation where you created two sad songs for me. Can you remind me what was the chord progression for the chorus in the second song?": ["chord progression for chorus second song", "second song sad songs chord progression"], "I\'m going back to our previous conversation about traditional Indian embroidery and tailoring techniques. Can you remind me of the name of that online store based in India that sells traditional Indian fabrics, threads, and embellishments?": ["online store in India that sells traditional Indian fabrics", "traditional Indian embroidery and tailoring techniques conversation", "Indian fabrics and threads store name", "embellishments for traditional Indian clothing retailer"], "I was looking back at our previous conversation about environmentally responsible supply chain practices, and I was wondering if you could remind me of the company you mentioned that\'s doing a great job with sustainability?": ["company with sustainable supply chain practices", "environmentally responsible supply chain example", "sustainability in supply chain conversation", "green supply chain case study"], "I was going through our previous conversation about high-end fashion brands, and I was wondering if you could remind me of the brand that uses wild rubber sourced from the Amazon rainforest?": ["high-end fashion brand with wild rubber from Amazon rainforest", "fashion brand using rubber from Amazon rainforest", "brand that uses wild rubber from Amazon rainforest"], "I remember you told me to dilute tea tree oil with a carrier oil before applying it to my skin. Can you remind me what the recommended ratio is?": ["dilute tea tree oil with carrier oil ratio", "tea tree oil carrier oil mixture recommendation", "how much tea tree oil to mix with carrier oil for skin", "recommended tea tree oil and carrier oil ratio for topical use"], "I was looking back at our previous conversation about Caribbean dishes and I was wondering, what was the name of that Jamaican dish you recommended I try with snapper that has fruit in it?": ["Jamaican dish with fruit and snapper", "Caribbean dishes conversation", "snapper with fruit dish recommendation", "Jamaican food with fruit and fish"]}')
DATA = Path("longmemeval_s_cleaned.json")
if not DATA.exists():
    urllib.request.urlretrieve("https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/longmemeval_s_cleaned.json", DATA)
convs = load_longmemeval(DATA)
cache = EmbeddingCache(sentence_transformers_encoder("BAAI/bge-small-en-v1.5", device="cuda"), max_chars=512, batch_size=512)
t = time.time(); cache.get([m.content for c in convs for m in c.messages]); print(f"embedded in {time.time()-t:.0f}s", flush=True)
QP = "Represent this sentence for searching relevant passages: "

class ExpandedHybrid(RecentPlusHybrid):
    def __init__(self, w0, **kw):
        super().__init__(cache, query_prefix=QP, **kw)
        self.w0, self.name = w0, f"hybrid+expansion w0={w0:g}"
    def rank(self, query, archive):
        score = defaultdict(float)
        for q, w in [(query, self.w0)] + [(s, 1.0) for s in EXP.get(query, [])]:
            for r, m in enumerate(super().rank(q, archive)):
                score[m.id] += w / (self.k + r + 1)
        by = {m.id: m for m in archive}
        return [by[i] for i in sorted(score, key=lambda i: -score[i])]

ads = [lambda: RecentPlusHybrid(cache, query_prefix=QP), lambda: ExpandedHybrid(1.0), lambda: ExpandedHybrid(2.0)]
t = time.time(); res = run(ads, convs, [4000, 16000]); print(f"ran in {time.time()-t:.0f}s")
pd.DataFrame([r.__dict__ for r in res]).to_csv("/kaggle/working/hybrid_expansion_results.csv", index=False)
DATA.unlink()
print(pd.DataFrame([r.__dict__ for r in res]).groupby(["adapter", "budget"])[["recalled", "recalled_all"]].mean().round(3))